<a href="https://colab.research.google.com/github/demonayush11/storyteller-from-scratch/blob/main/Storyteller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset

# Load TinyStories
ds = load_dataset("roneneldan/TinyStories")

# Start with 10k stories
stories = ds["train"]["text"][:10000]

# Merge into one text corpus
raw_text = "\n<|endoftext|>\n".join(stories)

print("Total characters:", len(raw_text))
print(raw_text[:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Total characters: 8824746
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [3]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['One', 'day', ',', 'a', 'little', 'girl', 'named', 'Lily', 'found', 'a', 'needle', 'in', 'her', 'room', '.', 'She', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', '.', 'Lily']


In [4]:
print(len(preprocessed))


2053627


In [5]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

10980


In [6]:
vocab = {token:integer for integer,token in enumerate(all_words)}


In [7]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
('$10', 2)
('&', 3)
("'", 4)
('*poof*', 5)
(',', 6)
('-', 7)
('--', 8)
('.', 9)
('1', 10)
('10', 11)
('123', 12)
('15', 13)
('16', 14)
('164', 15)
('2', 16)
('20', 17)
('3', 18)
('3-year', 19)
('3-year-old', 20)
('3-years-old', 21)
('30', 22)
('3rd', 23)
('4', 24)
('5', 25)
('50', 26)
('76', 27)
('8', 28)
('911', 29)
(':', 30)
(';', 31)
('<|endoftext|>', 32)
('?', 33)
('A', 34)
('ABC', 35)
('ABCs', 36)
('Abbie', 37)
('Abby', 38)
('Abi', 39)
('Abigail', 40)
('Abracadabra', 41)
('Absolutely', 42)
('Acceptor', 43)
('Accidents', 44)
('Achoo', 45)
('Across', 46)
('Acting', 47)
('Adam', 48)
('Add', 49)
('Adding', 50)


In [8]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [9]:
tokenizer = SimpleTokenizerV1(vocab)

text = """
Once upon a time there was a little dragon who lived in a cave.
"""

ids = tokenizer.encode(text)
print(ids)

[1233, 10146, 1972, 9754, 9632, 10318, 1972, 6142, 4112, 10465, 6146, 5640, 1972, 3091, 9]


In [10]:
tokenizer.decode(ids)


'Once upon a time there was a little dragon who lived in a cave.'

In [11]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [12]:
len(vocab.items())


10981

In [13]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('â€˜yesâ€™', 10976)
('â€“', 10977)
('â€”', 10978)
('â€™', 10979)
('<|unk|>', 10981)


In [14]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [15]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [16]:
tokenizer.encode(text)


[722,
 6,
 4045,
 10683,
 6107,
 9548,
 33,
 10980,
 786,
 9622,
 10981,
 10981,
 6844,
 9622,
 7014,
 9]

In [17]:
tokenizer.decode(tokenizer.encode(text))

'Hello, do you like tea? <|endoftext|> In the <|unk|> <|unk|> of the palace.'

In [18]:
! pip3 install tiktoken

In [19]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [20]:
tokenizer = tiktoken.get_encoding("gpt2")

In [21]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
